# Options Data Pipeline

Fetches **actual option prices** from Yahoo Finance for S&P 500 ETF and major underlying companies.

## Data Source
- **API:** yfinance (`ticker.option_chain(expiration_date)`)
- **Tickers:** SPY, AAPL, MSFT, GOOGL, AMZN, META
- **Target:** ~100 rows per ticker (calls + puts, various strikes and expirations)

## Output
- **Table:** `default.actual_option_prices`
- **Mode:** Overwrite (fresh snapshot each run)

## Schema
| Column | Type | Description |
| --- | --- | --- |
| ticker | string | Underlying ticker symbol |
| expiration_date | date | Option expiration date |
| days_to_expiry | int | Trading days until expiration |
| option_type | string | 'call' or 'put' |
| strike | double | Strike price (K) |
| last_price | double | Last traded option price |
| bid | double | Bid price |
| ask | double | Ask price |
| mid_price | double | (bid + ask) / 2 |
| implied_volatility | double | Implied volatility |
| volume | int | Trading volume |
| open_interest | int | Open interest |
| underlying_price | double | Current underlying price (S0) |
| fetched_at | timestamp | When data was fetched |

In [0]:
%pip install yfinance --quiet
dbutils.library.restartPython()

In [0]:
import sys
import time
from datetime import datetime, date

import numpy as np
import pandas as pd
import yfinance as yf

from pyspark.sql.functions import current_timestamp, lit

# Configuration
TICKERS = ["SPY", "AAPL", "MSFT", "GOOGL", "AMZN", "META"]
TARGET_TABLE = "default.actual_option_prices"

# Filter: only keep options expiring within this range (trading days)
MIN_DAYS_TO_EXPIRY = 5
MAX_DAYS_TO_EXPIRY = 60

# Max rows per ticker (calls + puts combined)
MAX_ROWS_PER_TICKER = 120

# Minimum volume/OI filter to avoid stale quotes
MIN_VOLUME = 1
MIN_OPEN_INTEREST = 10

print(f"Tickers: {TICKERS}")
print(f"Target table: {TARGET_TABLE}")
print(f"Expiry range: {MIN_DAYS_TO_EXPIRY}-{MAX_DAYS_TO_EXPIRY} trading days")
print(f"Max rows per ticker: {MAX_ROWS_PER_TICKER}")

In [0]:
# Per-chain rate limiting and a hard cap to avoid bursting the Yahoo API.
PER_CHAIN_SLEEP_SECONDS = 0.5
MAX_CHAINS_PER_TICKER = 12


def fetch_options_for_ticker(ticker_symbol: str, max_rows: int = 120) -> pd.DataFrame:
    """Fetch option chain data for a ticker from Yahoo Finance.

    Returns a DataFrame with calls and puts, filtered by liquidity and expiry range.
    """
    try:
        ticker = yf.Ticker(ticker_symbol)

        # Get underlying price
        hist = ticker.history(period="5d")
        if hist.empty:
            print(f"  [WARN] No price data for {ticker_symbol}, skipping.")
            return pd.DataFrame()
        underlying_price = float(hist["Close"].iloc[-1])

        # Get available expiration dates
        expirations = ticker.options
        if not expirations:
            print(f"  [WARN] No options available for {ticker_symbol}, skipping.")
            return pd.DataFrame()

        today = date.today()
        all_rows = []
        chains_queried = 0

        for exp_date_str in expirations:
            exp_date = datetime.strptime(exp_date_str, "%Y-%m-%d").date()
            # Cast both ends to numpy datetime64 explicitly so the result
            # is stable across NumPy versions and never auto-coerces.
            days_to_exp = int(
                np.busday_count(np.datetime64(today), np.datetime64(exp_date))
            )

            # Filter by expiry range
            if days_to_exp < MIN_DAYS_TO_EXPIRY or days_to_exp > MAX_DAYS_TO_EXPIRY:
                continue

            # Hard cap on Yahoo calls per ticker to avoid throttling.
            if chains_queried >= MAX_CHAINS_PER_TICKER:
                print(
                    f"  [INFO] Reached MAX_CHAINS_PER_TICKER={MAX_CHAINS_PER_TICKER} "
                    f"for {ticker_symbol}; stopping early."
                )
                break

            # Fetch option chain for this expiration
            try:
                chain = ticker.option_chain(exp_date_str)
            except Exception as e:
                print(f"  [WARN] Failed to get chain for {ticker_symbol} {exp_date_str}: {e}")
                continue
            finally:
                # Sleep between chain requests, not just between tickers.
                chains_queried += 1
                time.sleep(PER_CHAIN_SLEEP_SECONDS)

            # Process calls and puts
            for option_type, df in [("call", chain.calls), ("put", chain.puts)]:
                if df.empty:
                    continue

                # Filter by liquidity
                df = df[
                    (df["volume"].fillna(0) >= MIN_VOLUME) &
                    (df["openInterest"].fillna(0) >= MIN_OPEN_INTEREST) &
                    (df["lastPrice"] > 0)
                ].copy()

                # Filter strikes: within 20% of underlying
                lower = underlying_price * 0.80
                upper = underlying_price * 1.20
                df = df[(df["strike"] >= lower) & (df["strike"] <= upper)]

                for _, row in df.iterrows():
                    bid = float(row.get("bid", 0) or 0)
                    ask = float(row.get("ask", 0) or 0)
                    mid = (bid + ask) / 2 if (bid > 0 and ask > 0) else float(row["lastPrice"])

                    all_rows.append({
                        "ticker": ticker_symbol,
                        "expiration_date": exp_date,
                        "days_to_expiry": int(days_to_exp),
                        "option_type": option_type,
                        "strike": float(row["strike"]),
                        "last_price": float(row["lastPrice"]),
                        "bid": bid,
                        "ask": ask,
                        "mid_price": mid,
                        "implied_volatility": float(row.get("impliedVolatility", 0) or 0),
                        "volume": int(row.get("volume", 0) or 0),
                        "open_interest": int(row.get("openInterest", 0) or 0),
                        "underlying_price": underlying_price,
                    })

            # Stop early if we have enough rows
            if len(all_rows) >= max_rows:
                break

        result_df = pd.DataFrame(all_rows[:max_rows])
        print(f"  [OK] {ticker_symbol}: {len(result_df)} rows "
              f"(S0=${underlying_price:.2f}, {len(set(r['days_to_expiry'] for r in all_rows[:max_rows]))} expirations)")
        return result_df

    except Exception as e:
        print(f"  [ERROR] {ticker_symbol}: {e}")
        return pd.DataFrame()

print("Function defined.")

In [0]:
# ---------------------------------------------------------------------------
# Fetch option data for all tickers
# ---------------------------------------------------------------------------
print("=" * 60)
print("FETCHING OPTION CHAINS")
print("=" * 60)

all_options = []
for tkr in TICKERS:
    print(f"\n  Fetching {tkr}...")
    df = fetch_options_for_ticker(tkr, max_rows=MAX_ROWS_PER_TICKER)
    if not df.empty:
        all_options.append(df)
    time.sleep(1)  # rate limiting

if all_options:
    options_pdf = pd.concat(all_options, ignore_index=True)
    print(f"\n{'='*60}")
    print(f"TOTAL: {len(options_pdf)} rows across {options_pdf['ticker'].nunique()} tickers")
    print(f"{'='*60}")
    print(f"\nBreakdown:")
    print(options_pdf.groupby(["ticker", "option_type"]).size().unstack(fill_value=0).to_string())
    print(f"\nDays to expiry range: {options_pdf['days_to_expiry'].min()} - {options_pdf['days_to_expiry'].max()}")
    print(f"Strike range: ${options_pdf['strike'].min():.2f} - ${options_pdf['strike'].max():.2f}")
else:
    raise ValueError("No option data fetched! Check network connectivity.")

In [0]:
# ---------------------------------------------------------------------------
# Write to Delta table
# ---------------------------------------------------------------------------
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, IntegerType,
    DoubleType, TimestampType,
)

# NOTE: `fetched_at` is added below via withColumn(current_timestamp()).
# It MUST be declared in the schema or strict Delta enforcement will reject
# the write (and silent-drop in lenient mode would lose the column).
schema = StructType([
    StructField("ticker", StringType(), False),
    StructField("expiration_date", DateType(), False),
    StructField("days_to_expiry", IntegerType(), False),
    StructField("option_type", StringType(), False),
    StructField("strike", DoubleType(), False),
    StructField("last_price", DoubleType(), False),
    StructField("bid", DoubleType(), True),
    StructField("ask", DoubleType(), True),
    StructField("mid_price", DoubleType(), False),
    StructField("implied_volatility", DoubleType(), True),
    StructField("volume", IntegerType(), True),
    StructField("open_interest", IntegerType(), True),
    StructField("underlying_price", DoubleType(), False),
    StructField("fetched_at", TimestampType(), True),
])

# Convert dates to proper format
options_pdf["expiration_date"] = pd.to_datetime(options_pdf["expiration_date"]).dt.date

# Pre-populate fetched_at on the pandas frame so it lines up with the schema
# at createDataFrame time; current_timestamp() in Spark would also work, but
# keeping it on the pandas side avoids a separate withColumn round-trip.
options_pdf["fetched_at"] = pd.Timestamp.utcnow()

df = spark.createDataFrame(options_pdf, schema=schema)

# Write to Delta (overwrite for fresh snapshot)
df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)

row_count = spark.table(TARGET_TABLE).count()
print(f"\n[OK] Written {row_count} rows to {TARGET_TABLE}")
print(f"\nSample data:")
display(spark.table(TARGET_TABLE).limit(10))